# Despesas por placa — Maio/2026

Separa **todas as despesas** de maio/2026 (`02-Referencias/Fechamento/dados_maio_2026.xlsx`) cujo centro de custo (`descen`) contém uma das placas/identificadores solicitados nas demandas (imagens anexas).

## Saídas
- `outputs/tabelas/despesas_placas_maio_2026.xlsx` — abas `despesas`, `resumo_placa`, `resumo_grupo`, `catalogo_placas`
- `outputs/tabelas/despesas_placas_maio_2026.csv` — mesma base da aba `despesas` (UTF-8)

## Uso
1. Confirme que `dados_maio_2026.xlsx` está atualizado.
2. Execute **Run All**.
3. Placas sem lançamento em maio aparecem no resumo com quantidade zero.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
from IPython.display import display

# NOTEBOOK_DIR representa o diretório atual de onde o notebook está sendo executado.
# Esse valor normalmente será '04-Notebooks/Fechamento' se você estiver rodando o notebook de lá.
NOTEBOOK_DIR = Path.cwd()

# O _candidatos_root é uma lista de caminhos possíveis onde a pasta "02-Referencias" pode estar localizada.
# Ele tenta considerar diferentes lugares comuns a depender de onde você está executando este notebook.
# Explicando as opções:
# 1. NOTEBOOK_DIR / ".." / ".." / "02-Referencias": vai dois níveis para cima e tenta achar "02-Referencias" ali.
#    Ex: se NOTEBOOK_DIR = 04-Notebooks/Fechamento, então ".." = 04-Notebooks, ".." de novo = [repo root], então fica [repo root]/02-Referencias
# 2. NOTEBOOK_DIR / ".." / "02-Referencias": um nível para cima e procura "02-Referencias" ali (Ex: 04-Notebooks/02-Referencias)
# 3. Path("02-Referencias"): relativo ao diretório atual.
_candidatos_root = [
    NOTEBOOK_DIR / ".." / ".." / "02-Referencias",
    NOTEBOOK_DIR / ".." / "02-Referencias",
    Path("02-Referencias"),
]

# Aqui escolhe o primeiro desses caminhos da lista que realmente existe no sistema de arquivos.
# O método .resolve() transforma o caminho em absoluto, e .exists() verifica se a pasta existe.
REFS_DIR = next((p.resolve() for p in _candidatos_root if p.resolve().exists()), None)

# Se nenhum dos caminhos existir, para a execução e avisa o usuário.
if REFS_DIR is None:
    raise FileNotFoundError("Pasta 02-Referencias não encontrada (execute a partir do repo ou de 04-Notebooks/Fechamento).")

# ROOT é a raiz do projeto. No nosso caso, é o diretório pai da pasta 02-Referencias.
ROOT = REFS_DIR.parent

# ARQUIVO_ENTRADA é o caminho completo para o arquivo Excel com os dados de entrada.
ARQUIVO_ENTRADA = REFS_DIR / "dados_maio_2026.xlsx"
MES_REFERENCIA = "Maio/2026"

# OUT_DIR é o caminho onde vamos salvar as saídas (planilhas e csv).
OUT_DIR = ROOT / "outputs" / "tabelas"
OUT_XLSX = OUT_DIR / "despesas_placas_maio_2026.xlsx"
OUT_CSV = OUT_DIR / "despesas_placas_maio_2026.csv"

# Cria os diretórios de saída se ainda não existem.
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Verifica se o arquivo de entrada existe. Se não, para tudo e mostra uma mensagem clara.
if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(ARQUIVO_ENTRADA.resolve())

# Mostra os caminhos finais de entrada e saída para conferência do usuário.
print(f"Entrada: {ARQUIVO_ENTRADA.resolve()}")
print(f"Saída:   {OUT_XLSX.resolve()}")

Entrada: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\dados_maio_2026.xlsx
Saída:   C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\outputs\tabelas\despesas_placas_maio_2026.xlsx


## Catálogo de placas (demandas anexas)

Lista extraída das solicitações: máquinas alugadas (fechamento), frota G&S, lote Volvo/Ford/Hyundai/Liebherr e contratos IMAVI/BUSA/HIDROEUROPA.

In [2]:
# grupo_demanda | placa (como na planilha) | marca_modelo (referência da demanda)
_LINHAS_CATALOGO = [
    # Máquinas alugadas — fechamento maio/2026
    ("maquinas_alugadas_fechamento", "EHH0015", "HYUNDAI COM PENEIRA — ARCELOR BARRA MANSA"),
    ("maquinas_alugadas_fechamento", "EHH0037", "HYUNDAI COM GARRA E ÍMÃ — ARCELOR BARRA MANSA"),
    ("maquinas_alugadas_fechamento", "EHH0044", "HYUNDAI COM TESOURA — TUPY JOINVILLE/SC"),
    ("maquinas_alugadas_fechamento", "EHL0041", "LIEBHERR COM GARRA — TUPY JOINVILLE/SC"),
    ("maquinas_alugadas_fechamento", "PHH0049", "PRENSA MÓVEL — ARCELOR PIRACICABA"),
    ("maquinas_alugadas_fechamento", "PHH0044", "PRENSA MÓVEL — ARCELOR RESENDE X PIRA"),
    # Veículos — frota G&S (despesas 05/2026)
    ("veiculos_frota_gs", "NRQ7727", "MERCEDES-BENZ"),
    ("veiculos_frota_gs", "NRV7727", "MERCEDES-BENZ"),
    ("veiculos_frota_gs", "OON3B85", "VOLVO"),
    ("veiculos_frota_gs", "FBU7329", "VOLVO"),
    ("veiculos_frota_gs", "FJQ5A08", "FORD"),
    ("veiculos_frota_gs", "GHI3A74", "VOLVO"),
    ("veiculos_frota_gs", "GGE5059", "FORD"),
    ("veiculos_frota_gs", "TKI2E74", "VOLVO"),
    ("veiculos_frota_gs", "GDG8E75", "VOLVO"),
    ("veiculos_frota_gs", "BKW9I57", "VOLVO"),
    ("veiculos_frota_gs", "GDA9J63", "VOLVO"),
    ("veiculos_frota_gs", "BZL7H95", "VOLVO"),
    ("veiculos_frota_gs", "BTZ8D36", "VOLVO"),
    ("veiculos_frota_gs", "BZG6A91", "VOLVO"),
    ("veiculos_frota_gs", "SUY4F83", "VOLVO"),
    # Veículos / máquinas — lote 2
    ("veiculos_maquinas_lote2", "QAO4A53", "VOLVO"),
    ("veiculos_maquinas_lote2", "EWU6A85", "VOLVO"),
    ("veiculos_maquinas_lote2", "FRT7548", "VOLVO"),
    ("veiculos_maquinas_lote2", "EJZ7J97", "FORD"),
    ("veiculos_maquinas_lote2", "EWU6A79", "VOLVO"),
    ("veiculos_maquinas_lote2", "EWU5857", "VOLVO"),
    ("veiculos_maquinas_lote2", "GDY0H74", "VOLVO"),
    ("veiculos_maquinas_lote2", "GDB3A42", "VOLVO"),
    ("veiculos_maquinas_lote2", "EHH0019", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHH0006", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHH0008", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHH0007", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHH0004", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHH0005", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHL0014", "LIEBHERR"),
    ("veiculos_maquinas_lote2", "EHL0043", "LIEBHERR"),
    ("veiculos_maquinas_lote2", "EHL0041", "LIEBHERR"),
    ("veiculos_maquinas_lote2", "EHL0070", "LIEBHERR"),
    ("veiculos_maquinas_lote2", "EHH0038", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHH0036", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHH0040", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHH0046", "HYUNDAI"),
    ("veiculos_maquinas_lote2", "EHH0039", "HYUNDAI"),
    # Máquinas / contratos — lote 3
    ("maquinas_contratos_lote3", "EHH0037", "HYUNDAI"),
    ("maquinas_contratos_lote3", "EHH0002", "HYUNDAI"),
    ("maquinas_contratos_lote3", "EHH0015", "HYUNDAI"),
    ("maquinas_contratos_lote3", "EHH0003", "HYUNDAI"),
    ("maquinas_contratos_lote3", "EWU6341", "IMAVI"),
    ("maquinas_contratos_lote3", "EWU6294", "IMAVI"),
    ("maquinas_contratos_lote3", "FTP7D76", "BUSA"),
    ("maquinas_contratos_lote3", "EWU6154", "IMAVI"),
    ("maquinas_contratos_lote3", "EWU6204", "IMAVI"),
    ("maquinas_contratos_lote3", "EWU6343", "IMAVI"),
    ("maquinas_contratos_lote3", "EWU6B52", "IMAVI"),
    ("maquinas_contratos_lote3", "QAU6E20", "IMAVI"),
    ("maquinas_contratos_lote3", "FUP3G76", "BUSA"),
    ("maquinas_contratos_lote3", "FVQ3B78", "BUSA"),
    ("maquinas_contratos_lote3", "EHL0042", "LIEBHERR"),
    ("maquinas_contratos_lote3", "SWQ2C15/0049", "HIDROEUROPA"),
    ("maquinas_contratos_lote3", "GBL8H44/0044", "HIDROEUROPA"),
    ("maquinas_contratos_lote3", "QXH2G14/0061", "HIDROEUROPA"),
    ("maquinas_contratos_lote3", "PHH0062", "HIDROEUROPA"),
]

catalogo = pd.DataFrame(
    _LINHAS_CATALOGO,
    columns=["grupo_demanda", "placa", "marca_modelo"],
)
catalogo["token_busca"] = catalogo["placa"].str.split("/").str[0].str.strip().str.upper()

print(f"Placas no catálogo: {len(catalogo)} ({catalogo['token_busca'].nunique()} tokens únicos)")
display(catalogo.groupby("grupo_demanda").size().rename("qtd_placas"))

Placas no catálogo: 63 (60 tokens únicos)


grupo_demanda
maquinas_alugadas_fechamento     6
maquinas_contratos_lote3        19
veiculos_frota_gs               15
veiculos_maquinas_lote2         23
Name: qtd_placas, dtype: int64

In [3]:
def token_placa(placa: str) -> str:
    return str(placa).strip().split("/")[0].upper()


def placa_no_texto(texto: str, token: str) -> bool:
    """Evita falso positivo (ex.: PHH0044 dentro de PHH00445)."""
    if not token:
        return False
    padrao = r"(?<![A-Z0-9])" + re.escape(token) + r"(?![A-Z0-9])"
    return bool(re.search(padrao, str(texto).upper()))


def split_descen(descen: str) -> tuple[str, str, str, str]:
    partes = [p.strip() for p in str(descen).split("/") if p.strip()]
    natureza = partes[0] if partes else ""
    divisao = partes[1] if len(partes) > 1 else ""
    filial_cc = partes[2] if len(partes) > 2 else ""
    identificador = partes[-1] if partes else ""
    return natureza, divisao, filial_cc, identificador


def identificar_placas(descen: str, cat: pd.DataFrame) -> list[str]:
    hits: list[str] = []
    for _, row in cat.iterrows():
        if placa_no_texto(descen, row["token_busca"]):
            hits.append(row["placa"])
    # dedupe preservando ordem
    seen: set[str] = set()
    out: list[str] = []
    for p in hits:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def grupos_das_placas(placas: list[str], cat: pd.DataFrame) -> list[str]:
    if not placas:
        return []
    sub = cat[cat["placa"].isin(placas)]
    return sorted(sub["grupo_demanda"].unique().tolist())

In [4]:
df = pd.read_excel(ARQUIVO_ENTRADA)
df = df.loc[:, [c for c in df.columns if str(c).strip() and not str(c).startswith("Unnamed")]]

if "descen" not in df.columns:
    raise KeyError(f"Coluna 'descen' não encontrada. Colunas: {list(df.columns)}")

partes = df["descen"].astype(str).apply(split_descen)
df["natureza"] = partes.apply(lambda x: x[0])
df["divisao_cc"] = partes.apply(lambda x: x[1])
df["filial_cc"] = partes.apply(lambda x: x[2])
df["identificador_cc"] = partes.apply(lambda x: x[3])

df["placas_encontradas"] = df["descen"].apply(lambda d: identificar_placas(d, catalogo))
df["qtd_placas"] = df["placas_encontradas"].str.len()
df["placa_principal"] = df["placas_encontradas"].apply(lambda xs: xs[0] if xs else pd.NA)
df["grupos_demanda"] = df["placas_encontradas"].apply(lambda xs: grupos_das_placas(xs, catalogo))
df["grupos_demanda_txt"] = df["grupos_demanda"].apply(lambda g: "; ".join(g))
df["placas_txt"] = df["placas_encontradas"].apply(lambda xs: "; ".join(xs))

mask_despesa = df["natureza"].str.upper().eq("DESPESA")
mask_placa = df["qtd_placas"] > 0

despesas = df.loc[mask_despesa & mask_placa].copy()
despesas = despesas.sort_values(["placa_principal", "descdc", "lancamento"], na_position="last")

valor_col = "valor_centro" if "valor_centro" in despesas.columns else "valor_plano"

print(f"Linhas no arquivo: {len(df):,}")
print(f"Despesas (natureza=DESPESA): {mask_despesa.sum():,}")
print(f"Despesas das placas solicitadas: {len(despesas):,}")
print(f"Total {valor_col}: {despesas[valor_col].sum():,.2f}")

Linhas no arquivo: 5,959
Despesas (natureza=DESPESA): 738
Despesas das placas solicitadas: 337
Total valor_centro: -372,372.45


## Resumos

In [5]:
# Uma linha por placa no catálogo (mesma placa pode aparecer em mais de um grupo de demanda)
_catalogo_placas = (
    catalogo.groupby("placa", as_index=False)
    .agg(
        grupo_demanda=("grupo_demanda", lambda s: "; ".join(sorted(set(s)))),
        marca_modelo=("marca_modelo", "first"),
        token_busca=("token_busca", "first"),
    )
)

resumo_placa = (
    _catalogo_placas
    .merge(
        despesas.explode("placas_encontradas")
        .groupby("placas_encontradas")
        .agg(
            qtd_lancamentos=(valor_col, "size"),
            total_valor_centro=(valor_col, "sum"),
        )
        .rename_axis("placa")
        .reset_index(),
        on="placa",
        how="left",
    )
)
resumo_placa["qtd_lancamentos"] = resumo_placa["qtd_lancamentos"].fillna(0).astype(int)
resumo_placa["total_valor_centro"] = resumo_placa["total_valor_centro"].fillna(0.0)
resumo_placa = resumo_placa.sort_values(["grupo_demanda", "placa"])

# Totais por grupo de demanda (lançamento pode contar em mais de um grupo se bater várias placas)
_tmp_grupo = despesas[["grupos_demanda", valor_col]].explode("grupos_demanda")
resumo_grupo = (
    _tmp_grupo.groupby("grupos_demanda", as_index=False)
    .agg(
        qtd_lancamentos=(valor_col, "size"),
        total_valor_centro=(valor_col, "sum"),
    )
    .rename(columns={"grupos_demanda": "grupo_demanda"})
    .sort_values("grupo_demanda")
)
_catalogo_por_grupo = catalogo.groupby("grupo_demanda")["placa"].nunique().rename("qtd_placas_catalogo")
resumo_grupo = resumo_grupo.merge(
    _catalogo_por_grupo.reset_index(), on="grupo_demanda", how="left"
)

sem_lancamento = resumo_placa.loc[resumo_placa["qtd_lancamentos"] == 0, ["grupo_demanda", "placa", "marca_modelo"]]

print("Resumo por grupo de demanda:")
display(resumo_grupo)

if len(sem_lancamento):
    print(f"\nPlacas sem despesa em {MES_REFERENCIA} ({len(sem_lancamento)}):")
    display(sem_lancamento)
else:
    print("\nTodas as placas do catálogo tiveram pelo menos um lançamento.")

print("\nTop 15 placas por valor absoluto:")
_top = resumo_placa.assign(_abs=lambda d: d["total_valor_centro"].abs()).nlargest(15, "_abs")
display(
    _top[["grupo_demanda", "placa", "marca_modelo", "qtd_lancamentos", "total_valor_centro"]]
)

Resumo por grupo de demanda:


,grupo_demanda,qtd_lancamentos,total_valor_centro,qtd_placas_catalogo
0,maquinas_alugadas_fechamento,34,-37329.73,6
1,maquinas_contratos_lote3,78,-109319.06,19
2,veiculos_frota_gs,155,-167787.16,15
3,veiculos_maquinas_lote2,98,-79266.61,23



Placas sem despesa em Maio/2026 (6):


,grupo_demanda,placa,marca_modelo
28,maquinas_contratos_lote3,EWU6204,IMAVI
34,maquinas_contratos_lote3,EWU6B52,IMAVI
48,veiculos_frota_gs,NRQ7727,MERCEDES-BENZ
6,veiculos_maquinas_lote2,EHH0004,HYUNDAI
7,veiculos_maquinas_lote2,EHH0005,HYUNDAI
10,veiculos_maquinas_lote2,EHH0008,HYUNDAI



Top 15 placas por valor absoluto:


,grupo_demanda,placa,marca_modelo,qtd_lancamentos,total_valor_centro
42,veiculos_frota_gs,GDA9J63,VOLVO,7,-28133.23
36,veiculos_frota_gs,FJQ5A08,FORD,10,-26074.03
44,veiculos_frota_gs,GDG8E75,VOLVO,34,-25411.38
0,veiculos_frota_gs,BKW9I57,VOLVO,6,-25084.70
22,maquinas_contratos_lote3,EHL0042,LIEBHERR,6,-23321.27
56,maquinas_contratos_lote3,QXH2G14/0061,HIDROEUROPA,21,-22262.14
47,veiculos_frota_gs,GHI3A74,VOLVO,30,-22178.97
1,veiculos_frota_gs,BTZ8D36,VOLVO,3,-17781.54
53,maquinas_contratos_lote3,PHH0062,HIDROEUROPA,9,-16784.51
12,veiculos_maquinas_lote2,EHH0019,HYUNDAI,6,-16508.35


## Máquinas alugadas (fechamento)

Recorte específico da primeira demanda (custos de locação para fechamento).

In [6]:
grupo_fechamento = "maquinas_alugadas_fechamento"
despesas_fechamento = despesas[despesas["grupos_demanda_txt"].str.contains(grupo_fechamento, na=False)].copy()

resumo_fechamento = (
    despesas_fechamento.groupby(["placa_principal", "descdc"], dropna=False)[valor_col]
    .agg(qtd=("size"), total=("sum"))
    .reset_index()
    .sort_values("total")
)

print(f"Linhas — {grupo_fechamento}: {len(despesas_fechamento)}")
print(f"Total {valor_col}: {despesas_fechamento[valor_col].sum():,.2f}")
display(resumo_fechamento)
display(
    despesas_fechamento[
        ["lancamento", "descen", "descdc", "nome", valor_col, "documento", "filial"]
    ].head(20)
)

Linhas — maquinas_alugadas_fechamento: 34
Total valor_centro: -37,329.73


,placa_principal,descdc,qtd,total
3,EHH0044,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,6,-15999.62
11,PHH0049,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,4,-7934.00
7,PHH0044,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,10,-7118.01
0,EHH0015,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,3,-2530.89
2,EHH0037,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,2,-2183.00
12,PHH0049,SEGURANÇA DO TRABALHO,1,-500.00
1,EHH0037,FRETES E CARRETOS,1,-353.55
6,EHL0041,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,2,-214.29
10,PHH0049,FRETES E CARRETOS,1,-160.61
4,EHL0041,CORREIOS,1,-95.26


,lancamento,descen,descdc,nome,valor_centro,documento,filial
4775,2026-05-05,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,BARRALUB BARRA MANSA OIL LTDA,-1091.50,NFE-37326,G&S PRUDENTE
4777,2026-05-05,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,BARRALUB BARRA MANSA OIL LTDA,-1091.50,NFE-37326,G&S PRUDENTE
3280,2026-05-14,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,ACO RUBER COMERCIAL LTDA,-347.89,NFE-71166,G&S PRUDENTE
1256,2026-05-15,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,FRETES E CARRETOS,RODONAVES TRANSP. E ENC. LTDA,-353.55,CTE30755/30921,G&S PRUDENTE
4774,2026-05-05,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,BARRALUB BARRA MANSA OIL LTDA,-1091.50,NFE-37326,G&S PRUDENTE
4776,2026-05-05,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,BARRALUB BARRA MANSA OIL LTDA,-1091.50,NFE-37326,G&S PRUDENTE
5494,2026-05-09,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,REI DOS PARAFUSOS LTDA,-443.81,NFE-15441,G3S PRUDENTE
1958,2026-05-11,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,DISTRIBUIDORA DE LUBRIFICANTES GRINGO LTDA,-170.00,NFE-1563180,G&S PRUDENTE
3395,2026-05-18,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,DISTRIBUIDORA DE LUBRIFICANTES GRINGO LTDA,-3450.00,NFE-1563436,G&S PRUDENTE
1190,2026-05-20,DESPESA / EKIPA LOCACOES E SERV G&S / CONTRAT...,MANUTENÇÃO DE VEÍCULOS/MAQUINAS,E C DOURADO ASSISTENCIA TECNICA,-6000.00,NFSE-4,G&S MARINGA


## Exportar

In [7]:
cols_export = [
    c
    for c in [
        "codcen",
        "descen",
        "natureza",
        "divisao_cc",
        "filial_cc",
        "identificador_cc",
        "placas_txt",
        "placa_principal",
        "grupos_demanda_txt",
        "codcdc",
        "descdc",
        "lancamento",
        "ite_pagrec_vencimento",
        "iterea_pagamento",
        "iterea_valpago",
        "documento",
        "codigo_pessoa",
        "nome",
        "valor_plano",
        "valor_centro",
        "valor_bruto",
        "observacao",
        "nota",
        "filial",
    ]
    if c in despesas.columns
]

export_df = despesas[cols_export].copy()
export_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    export_df.to_excel(writer, sheet_name="despesas", index=False)
    resumo_placa.to_excel(writer, sheet_name="resumo_placa", index=False)
    resumo_grupo.to_excel(writer, sheet_name="resumo_grupo", index=False)
    catalogo.to_excel(writer, sheet_name="catalogo_placas", index=False)
    despesas_fechamento[cols_export].to_excel(writer, sheet_name="maquinas_fechamento", index=False)
    sem_lancamento.to_excel(writer, sheet_name="placas_sem_lancamento", index=False)

print(f"CSV:  {OUT_CSV.resolve()}")
print(f"XLSX: {OUT_XLSX.resolve()}")

CSV:  C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\outputs\tabelas\despesas_placas_maio_2026.csv
XLSX: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\outputs\tabelas\despesas_placas_maio_2026.xlsx
